In [8]:
from Trips_Extractor import extract_trips_from_pdf
from Lines_Extractor import parse_line_report_pdf
from master_lines_creation import creating_master_line
from pprint import pprint
import Processing_fucntions as pf
import pandas as pd

In [9]:
lines_pdf = "/home/aleluc/Github_repos/UPS-project/SamplePDFs/2304 Lines edge case.pdf"
lines = parse_line_report_pdf(lines_pdf, first_calendar_page=1)
bid_period_info = {x: lines[x] for x in ('bid_period_date_range','pay_period_date_ranges')}

In [10]:
print(bid_period_info)

{'bid_period_date_range': {'start': '2023-05-21', 'end': '2023-07-16'}, 'pay_period_date_ranges': {'PP1': {'start': '2023-05-21', 'end': '2023-06-17'}, 'PP2': {'start': '2023-06-18', 'end': '2023-07-15'}}}


In [11]:
trips_pdf = "/home/aleluc/Github_repos/UPS-project/SamplePDFs/2304 Trips.pdf"
trips = extract_trips_from_pdf(trips_pdf)

In [12]:
master_lines = creating_master_line(trips, lines)

In [23]:
pf.add_blockiness_scores(master_lines, bid_period_info)
pf.add_company_ticket_percentages(master_lines)
pf.add_training_fit_score(master_lines, "2023-01-06", "2023-01-10",bid_period_info)
vacation_ranges=[{"start": "2023-05-01", "end": "2023-05-15"},{"start": "2023-06-01", "end": "2023-06-08"}]
new_vacation_ranges = pf.add_vacation_days_off_score(master_lines,vacation_ranges, bid_period_info,pp_drop_threshold_days=14,save_details=False)
pf.add_bid_edge_days_off(master_lines,bid_period_info,edge="start")

In [14]:
pprint(master_lines)

{1: {'PPs': [{'BT': '52:48',
              'CT': '72:00',
              'DD': 12,
              'DO': 11,
              'assignments': [{'flights': [{'arrival': 'MIA',
                                            'departure': 'SDF',
                                            'end_date': '2023-05-23',
                                            'rest': None,
                                            'route_flags': None,
                                            'start_date': '2023-05-23'},
                                           {'arrival': 'SDF',
                                            'departure': 'MIA',
                                            'end_date': '2023-05-23',
                                            'rest': None,
                                            'route_flags': None,
                                            'start_date': '2023-05-23'}],
                               'premium': 0.0,
                               'tafb': '7h58',
               

In [15]:
def build_line_calendar_values(line_data, bid_dates, off_value=""):
    """
    Creates the day-by-day calendar contents for one line.

    Returns:
        {
            date(2023, 6, 1): "{400SDF-ATL-[MDT27.8]",
            date(2023, 6, 2): "MDT-SDF-[MDT15.2]",
            ...
        }
    """

    RESERVE_CODES = {"VTO", "RB", "RA", "SA", "SB", "VOR"}

    def to_date(value):
        if value is None or value == "":
            return None
        if isinstance(value, datetime):
            return value.date()
        if isinstance(value, date):
            return value
        return datetime.strptime(str(value), "%Y-%m-%d").date()

    def date_range_strict(start, end):
        start = to_date(start)
        end = to_date(end)

        if start is None or end is None:
            return []

        if start > end:
            return []

        return [start + timedelta(days=i) for i in range((end - start).days + 1)]

    def format_rest(rest):
        if rest is None:
            return ""

        if isinstance(rest, (int, float)):
            return str(int(rest)) if float(rest).is_integer() else str(round(float(rest), 1))

        text = str(rest).strip()

        if "h" in text:
            hours, minutes = text.split("h", 1)
        elif ":" in text:
            hours, minutes = text.split(":", 1)
        else:
            return text

        hours = int(hours)
        minutes = int(minutes or 0)
        value = hours + minutes / 60

        return str(int(value)) if value.is_integer() else str(round(value, 1))

    def format_route_flags(route_flags):
        if not route_flags:
            return ""

        if isinstance(route_flags, str):
            flags = [route_flags]
        else:
            flags = list(route_flags)

        flags = [str(flag).strip() for flag in flags if str(flag).strip()]

        if not flags:
            return ""

        return f"({','.join(flags)})"

    def arrival_text(arrival, rest=None, close_trip=False):
        if rest is not None:
            text = f"[{arrival}{format_rest(rest)}] "
        else:
            text = str(arrival)

        if close_trip:
            text = text.rstrip() + "}"

        return text

    def append_piece(parts_by_date, d, piece):
        if not piece:
            return

        piece = str(piece)

        if piece == "*":
            if not parts_by_date[d]:
                parts_by_date[d] = "*"
            return

        if parts_by_date[d] == "*":
            parts_by_date[d] = piece
            return

        if not parts_by_date[d]:
            parts_by_date[d] = piece
            return

        if parts_by_date[d].endswith(" "):
            parts_by_date[d] += piece
        else:
            parts_by_date[d] += piece

    def render_trip(assignment):
        trip_id = assignment.get("trip_id")
        flights = assignment.get("flights") or []
        parts_by_date = defaultdict(str)

        previous_arrival = None
        previous_end_date = None
        previous_rest = None

        for index, flight in enumerate(flights):
            dep = flight.get("departure")
            arr = flight.get("arrival")

            start_date = to_date(flight.get("start_date"))
            end_date = to_date(flight.get("end_date"))

            if start_date is None and end_date is None:
                continue

            if start_date is None:
                start_date = end_date

            if end_date is None:
                end_date = start_date

            route_flags = format_route_flags(flight.get("route_flags"))
            rest = flight.get("rest")

            is_first_flight = index == 0
            is_last_flight = index == len(flights) - 1

            trip_open = f"{{{trip_id}" if is_first_flight else ""

            arrival = arrival_text(
                arr,
                rest=rest,
                close_trip=is_last_flight,
            )

            can_compress_departure = (
                not is_first_flight
                and previous_arrival == dep
                and previous_end_date == start_date
                and previous_rest is None
                and parts_by_date[start_date]
            )

            if start_date == end_date:
                if can_compress_departure:
                    piece = f"-{route_flags}{arrival}"
                else:
                    piece = f"{trip_open}{dep}-{route_flags}{arrival}"

                append_piece(parts_by_date, start_date, piece)

            else:
                if can_compress_departure:
                    departure_piece = "-"
                else:
                    departure_piece = f"{trip_open}{dep}-"

                arrival_piece = f"{route_flags}{arrival}"

                append_piece(parts_by_date, start_date, departure_piece)
                append_piece(parts_by_date, end_date, arrival_piece)

            if index < len(flights) - 1:
                next_flight = flights[index + 1]
                next_start = to_date(next_flight.get("start_date"))

                if next_start is not None:
                    gap_start = end_date + timedelta(days=1)
                    gap_end = next_start - timedelta(days=1)

                    for gap_day in date_range_strict(gap_start, gap_end):
                        append_piece(parts_by_date, gap_day, "*")

            previous_arrival = arr
            previous_end_date = end_date
            previous_rest = rest

        return parts_by_date

    def merge_pieces(pieces):
        cleaned = [str(p).strip() for p in pieces if p and str(p).strip()]

        if not cleaned:
            return off_value

        real_pieces = [p for p in cleaned if p != "*"]

        if real_pieces:
            return " ".join(dict.fromkeys(real_pieces))

        return chr(8212)

    text_by_date = defaultdict(list)

    for pp in line_data.get("PPs", line_data.get("pay_periods", [])):
        for assignment in pp.get("assignments", []):

            if assignment.get("flights"):
                rendered = render_trip(assignment)

            elif assignment.get("code") in RESERVE_CODES:
                assignment_date = to_date(assignment.get("date"))
                rendered = {assignment_date: assignment["code"]} if assignment_date else {}

            else:
                rendered = {}

            for d, text in rendered.items():
                if bid_dates[0] <= d <= bid_dates[-1]:
                    text_by_date[d].append(text)

    return {
        d: merge_pieces(text_by_date.get(d, []))
        for d in bid_dates
    }

In [16]:
from datetime import date, datetime, timedelta
from collections import defaultdict
def _date_range_inclusive(start, end):
    start = _to_date(start)
    end = _to_date(end)

    if start is None or end is None:
        return []

    if end < start:
        start, end = end, start

    return [
        start + timedelta(days=i)
        for i in range((end - start).days + 1)
    ]

def _to_date(value):
    if value is None or value == "":
        return None

    if isinstance(value, datetime):
        return value.date()

    if isinstance(value, date):
        return value

    return datetime.strptime(str(value), "%Y-%m-%d").date()

def master_lines_to_dataframe(
    master_lines,
    bid_period_info,
    *,
    date_col_format="%Y-%m-%d",
    start_bid_off = False,
    end_bid_off = False,
):

    bid_start = bid_period_info["bid_period_date_range"]["start"]
    bid_end = bid_period_info["bid_period_date_range"]["end"]
    bid_dates = _date_range_inclusive(bid_start, bid_end)

    include_start_bid_off = start_bid_off or any(
        "bid_start_days_off" in line_data
        for line_data in master_lines.values()
    )

    include_end_bid_off = end_bid_off or any(
        "bid_end_days_off" in line_data
        for line_data in master_lines.values()
    )

    rows = []

    for line_number, line_data in master_lines.items():

        date_values = build_line_calendar_values(
            line_data,
            bid_dates,
            off_value="",
        )
        
        row = {
            "Line Number": line_number,
            "Extra Vacation Days": line_data.get("extra_vacation_days", 0),
        }

        # Calendar date columns go here
        for d in bid_dates:
            row[d.strftime(date_col_format)] = date_values[d]

        # Score / sorting columns go after the calendar
        row.update({
            "Training": line_data.get("training_fit_score", 0),
            "Blockiness": line_data.get("blockiness_score", 0),
            "Total DO": line_data.get("tot_DO", 0),
            "% tickets paid": line_data.get("company_ticket_pct", 0),
            "Total CT": line_data.get("tot_CT",0),
            "Premium": int(line_data.get("tot_Premium", line_data.get("tot_premium", 0))),
        })

        if include_start_bid_off:
            row.update({"Start bid off":line_data.get("bid_start_days_off",0)})
        
        if include_end_bid_off:
            row.update({"End bid off":line_data.get("bid_end_days_off",0)})

        """
        row = {
            "Line Number": line_number,
            "Extra Vacation Days": line_data.get("extra_vacation_days", 0),
            "Training": line_data.get("training_fit_score", 0),
            "Blockiness": line_data.get("blockiness_score", 0),
            "Total DO": line_data.get("tot_DO", 0),
            "% tickets paid": line_data.get("company_ticket_pct", 0),
            "Premium": line_data.get("tot_premium")
        }

        for d in bid_dates:
            row[d.strftime(date_col_format)] = date_values[d]
        """
        rows.append(row)

    return pd.DataFrame(rows).sort_values("Line Number").reset_index(drop=True)

In [17]:
df = master_lines_to_dataframe(
    master_lines,
    bid_period_info,
)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)  # Shows full text in cells
df

,Line Number,Extra Vacation Days,2023-05-21,2023-05-22,2023-05-23,2023-05-24,2023-05-25,2023-05-26,2023-05-27,2023-05-28,2023-05-29,2023-05-30,2023-05-31,2023-06-01,2023-06-02,2023-06-03,2023-06-04,2023-06-05,2023-06-06,2023-06-07,2023-06-08,2023-06-09,2023-06-10,2023-06-11,2023-06-12,2023-06-13,2023-06-14,2023-06-15,2023-06-16,2023-06-17,2023-06-18,2023-06-19,2023-06-20,2023-06-21,2023-06-22,2023-06-23,2023-06-24,2023-06-25,2023-06-26,2023-06-27,2023-06-28,2023-06-29,2023-06-30,2023-07-01,2023-07-02,2023-07-03,2023-07-04,2023-07-05,2023-07-06,2023-07-07,2023-07-08,2023-07-09,2023-07-10,2023-07-11,2023-07-12,2023-07-13,2023-07-14,2023-07-15,2023-07-16,Training,Blockiness,Total DO,% tickets paid,Total CT,Premium,Start bid off
0,1,2,,,{7SDF-MIA-SDF},{7SDF-MIA-SDF},{14SDF-MCO-SDF},,,,,,{8SDF-MIA-SDF},{8SDF-MIA-SDF},,,,,{8SDF-MIA-SDF},{8SDF-MIA-SDF},{8SDF-MIA-SDF},{34SDF-PHL-SDF},,,,{8SDF-MIA-SDF},{8SDF-MIA-SDF},{8SDF-MIA-SDF},,,,,{22SDF-IAH-SDF},{8SDF-MIA-SDF},{8SDF-MIA-SDF},,,,,{8SDF-MIA-SDF},{8SDF-MIA-SDF},{8SDF-MIA-SDF},,,,,,,{9SDF-MIA-SDF},,,,,{9SDF-MIA-SDF},{9SDF-MIA-SDF},{9SDF-MIA-SDF},{34SDF-PHL-SDF},{46SDF-DTW-(DH DL)SDF},,0.0,704.748252,24,2.1,144.00,0,2
1,2,2,,,{4SDF-EWR-SDF},{4SDF-EWR-SDF},{4SDF-EWR-SDF},{4SDF-EWR-SDF},{31SDF-MKE-SDF},,,,,,,,,,{5SDF-EWR-SDF},{5SDF-EWR-SDF},{5SDF-EWR-SDF},,,,,{5SDF-EWR-SDF},{5SDF-EWR-SDF},{5SDF-EWR-SDF},{65SDF-EWR-SDF},,,,,,,,,,,{5SDF-EWR-SDF},{5SDF-EWR-SDF},{5SDF-EWR-SDF},{5SDF-EWR-SDF},,,,,{49SDF-JFK-SDF},{3SDF-JFK-SDF},{65SDF-EWR-SDF},,,,{6SDF-EWR-SDF},{6SDF-EWR-SDF},{6SDF-EWR-SDF},{6SDF-EWR-SDF},,,0.0,705.915064,25,0.0,138.57,0,2
2,37,0,{291SDF-,(DH DL)ATL-(DH DL)[DFW20.9],DFW-RFD-[DFW15.4],DFW-SDF-[OAK25.2],OAK-[ONT16.6],ONT-DEN-SDF},,,,,,,,,,,,,,,,{356SDF-,(DH AA)CLT-(DH AA)[DFW20.5],DFW-RFD-[DFW15.8],DFW-RFD-[DFW15.4],DFW-SDF-[OAK25.2],OAK-[ONT13.5],ONT-(DH DL)ATL-(DH DL)SDF},,,,,,,,{396SDF-(DH DL)DTW-,[LAN24.1],LAN-SDF-[LAN15.8],LAN-SDF-[LAN15.8],LAN-SDF-[LAN15.8],LAN-SDF-[LAN15.8],LAN-SDF-[EWR17.4],EWR-SDF},,,,,,,,{309SDF-(DH WN)MDW-(DH WN)[MCI20.5],MCI-SDF-[MKE15.3],MKE-SDF-[MKE15.3],MKE-SDF-[MKE15.3],MKE-SDF-[MKE15.3],MKE-SDF},,0.0,710.294639,28,62.5,140.05,0,0
3,38,0,{307SDF-,(DH WN)[ATL24.0],ATL-SDF-[CLE14.7],CLE-SDF-[CLE14.7],CLE-SDF-[CLE14.7],CLE-SDF-[CLE14.7],CLE-SDF},,,,,,,,{400SDF-(DH DL)ATL-(DH DL)[MDT27.8],—,MDT-SDF-[MDT15.2],MDT-SDF-[MDT15.2],MDT-SDF-[MDT15.2],MDT-SDF-[MDT15.9],MDT-SDF-[MDT11.6] MDT-,(DH AA)CLT-(DH AA)SDF},,,,,,,{326SDF-(DH DL)[DTW26],—,DTW-SDF-[ATL16.9],ATL-SDF-[ATL16.9],ATL-SDF-[ATL16.9],ATL-SDF-[MIA14.7],MIA-SDF},,{338SDF-(DH AA)[DFW14.8],DFW-[PHL31.6],PHL-,SDF-[ORD16.6],ORD-SDF-[MCO14.6],MCO-SDF-[CLE12.6] CLE-,(DH WN)MDW-(DH WN)SDF},,,,,,,,,,,,,,,0.0,708.584535,31,75.0,143.28,0,0
4,105,0,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,,,,{600SDF-,[HNL20.6],HNL-(DH DL)ATL-(DH DL)SDF},,,,,,{600SDF-,[HNL20.6],HNL-(DH DL)ATL-(DH DL)SDF},,,,,{601SDF-,[HNL20.6],HNL-(DH DL)ATL-(DH DL)SDF},,,,,,"{612SDF-(C,DH UPS)CGN-[FRA25.1]",FRA-,(DH EK)DXB-(DH EK)[BAH17.6] BAH-,0.0,684.184028,16,37.5,69.97,439,0
5,106,0,{395SDF-(DH DL)[ATL25.5],—,ATL-SDF-[ATL16.9],ATL-SDF-[ATL16.9],ATL-SDF-[ATL16.9],ATL-SDF-[MCO14.6],MCO-SDF-[EWR17.4],EWR-SDF},,,,,,,,,,,,,,{310SDF-,(DH AA)CLT-(DH AA)[RDU20.0],RDU-SDF-[RDU14.5],RDU-SDF-[RDU14.5],RDU-SDF-[RDU14.5],RDU-SDF-[RDU9.8] RDU-(DH AA)CLT-(DH AA)SDF},,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,,0.0,685.711538,13,75.0,74.40,0,0
6,135,0,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,,0.0,660.000000,0,0.0,0.00,0,0
7,136,0,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,VOR,

In [18]:
import pandas as pd


def sort_dataframe_by_conditions(
    df,
    sort_order,
    drop_all_zero_or_none_cols=True,
    reset_index=True,
    missing_col_action="raise",
):
    """
    Sorts a DataFrame using user-provided sorting conditions.

    Parameters
    ----------
    df:
        Pandas DataFrame.

    sort_order:
        List of sorting rules in priority order.

        Each rule should be either:

            ("Column Name", "desc")

        or:

            ("Column Name", "asc")

        You can also use:

            ("Column Name", "high_to_low")
            ("Column Name", "low_to_high")
            ("Column Name", False)   # False = descending
            ("Column Name", True)    # True = ascending

    drop_all_zero_or_none_cols:
        If True, any sorting column where all values are 0, None, NaN,
        or blank will be dropped from the DataFrame and removed from sorting.

    reset_index:
        If True, resets the index after sorting.

    missing_col_action:
        "raise" -> raise an error if a sorting column is missing.
        "ignore" -> skip missing sorting columns.
    """

    df = df.copy()

    def normalize_sort_direction(direction):
        """
        Returns True for ascending, False for descending.
        """
        if isinstance(direction, bool):
            return direction

        direction = str(direction).lower().strip()

        if direction in ["asc", "ascending", "low_to_high", "small_to_large", "smallest_to_largest"]:
            return True

        if direction in ["desc", "descending", "high_to_low", "large_to_small", "largest_to_smallest"]:
            return False

        raise ValueError(
            f"Invalid sort direction: {direction}. "
            "Use 'asc', 'desc', True, or False."
        )

    def numeric_col(col):
        """
        Converts column to numeric for sorting and zero-checking.

        Handles:
            10
            "10"
            "10%"
            ""
            None
            NaN
        """

        cleaned = (
            df[col]
            .astype(str)
            .str.replace("%", "", regex=False)
            .str.strip()
            .replace({
                "": None,
                "None": None,
                "nan": None,
                "NaN": None,
            })
        )

        return pd.to_numeric(cleaned, errors="coerce").fillna(0)

    active_sort_cols = []
    ascending_values = []

    for col, direction in sort_order:

        if col not in df.columns:
            if missing_col_action == "raise":
                raise KeyError(f"Missing sorting column: {col}")
            elif missing_col_action == "ignore":
                continue
            else:
                raise ValueError("missing_col_action must be 'raise' or 'ignore'.")

        col_numeric = numeric_col(col)

        all_zero_or_none = col_numeric.eq(0).all()

        if drop_all_zero_or_none_cols and all_zero_or_none:
            df = df.drop(columns=[col])
            continue

        active_sort_cols.append(col)
        ascending_values.append(normalize_sort_direction(direction))

    if not active_sort_cols:
        if reset_index:
            return df.reset_index(drop=True)
        return df

    temp_sort_cols = []

    for col in active_sort_cols:
        temp_col = f"__sort_{col}"

        while temp_col in df.columns:
            temp_col += "_"

        df[temp_col] = numeric_col(col)
        temp_sort_cols.append(temp_col)

    df = df.sort_values(
        by=temp_sort_cols,
        ascending=ascending_values,
        kind="mergesort",
    )

    df = df.drop(columns=temp_sort_cols)

    if reset_index:
        df = df.reset_index(drop=True)

    return df

In [19]:
sortorder = [("Extra Vacation Days","high_to_low"),("Training","high_to_low"),("Blockiness","high_to_low"),("Total DO","high_to_low"),("% tickets paid", "high_to_low"),("Premium","high_to_low")]

dfsorted = sort_dataframe_by_conditions(df, sortorder)
dfsorted

,Line Number,Extra Vacation Days,2023-05-21,2023-05-22,2023-05-23,2023-05-24,2023-05-25,2023-05-26,2023-05-27,2023-05-28,2023-05-29,2023-05-30,2023-05-31,2023-06-01,2023-06-02,2023-06-03,2023-06-04,2023-06-05,2023-06-06,2023-06-07,2023-06-08,2023-06-09,2023-06-10,2023-06-11,2023-06-12,2023-06-13,2023-06-14,2023-06-15,2023-06-16,2023-06-17,2023-06-18,2023-06-19,2023-06-20,2023-06-21,2023-06-22,2023-06-23,2023-06-24,2023-06-25,2023-06-26,2023-06-27,2023-06-28,2023-06-29,2023-06-30,2023-07-01,2023-07-02,2023-07-03,2023-07-04,2023-07-05,2023-07-06,2023-07-07,2023-07-08,2023-07-09,2023-07-10,2023-07-11,2023-07-12,2023-07-13,2023-07-14,2023-07-15,2023-07-16,Blockiness,Total DO,% tickets paid,Total CT,Premium,Start bid off
0,172,7,,,,,,,,SB,SB,SB,SB,SB,SB,SB,,,,,,,,SB,SB,SB,SB,SB,SB,SB,,,,,,,,SB,SB,SB,SB,SB,SB,SB,,,,,,,,SB,SB,SB,SB,SB,SB,SB,,307.266667,28,0.0,0.00,0,7
1,156,7,,,,,,,,SA,SA,SA,SA,SA,SA,SA,,,,,,,,SA,SA,SA,SA,SA,SA,SA,,,,,,,,SA,SA,SA,SA,SA,SA,SA,,,,,,,,SA,SA,SA,SA,SA,SA,SA,,207.266667,28,0.0,0.00,0,7
2,2,2,,,{4SDF-EWR-SDF},{4SDF-EWR-SDF},{4SDF-EWR-SDF},{4SDF-EWR-SDF},{31SDF-MKE-SDF},,,,,,,,,,{5SDF-EWR-SDF},{5SDF-EWR-SDF},{5SDF-EWR-SDF},,,,,{5SDF-EWR-SDF},{5SDF-EWR-SDF},{5SDF-EWR-SDF},{65SDF-EWR-SDF},,,,,,,,,,,{5SDF-EWR-SDF},{5SDF-EWR-SDF},{5SDF-EWR-SDF},{5SDF-EWR-SDF},,,,,{49SDF-JFK-SDF},{3SDF-JFK-SDF},{65SDF-EWR-SDF},,,,{6SDF-EWR-SDF},{6SDF-EWR-SDF},{6SDF-EWR-SDF},{6SDF-EWR-SDF},,,705.915064,25,0.0,138.57,0,2
3,1,2,,,{7SDF-MIA-SDF},{7SDF-MIA-SDF},{14SDF-MCO-SDF},,,,,,{8SDF-MIA-SDF},{8SDF-MIA-SDF},,,,,{8SDF-MIA-SDF},{8SDF-MIA-SDF},{8SDF-MIA-SDF},{34SDF-PHL-SDF},,,,{8SDF-MIA-SDF},{8SDF-MIA-SDF},{8SDF-MIA-SDF},,,,,{22SDF-IAH-SDF},{8SDF-MIA-SDF},{8SDF-MIA-SDF},,,,,{8SDF-MIA-SDF},{8SDF-MIA-SDF},{8SDF-MIA-SDF},,,,,,,{9SDF-MIA-SDF},,,,,{9SDF-MIA-SDF},{9SDF-MIA-SDF},{9SDF-MIA-SDF},{34SDF-PHL-SDF},{46SDF-DTW-(DH DL)SDF},,704.748252,24,2.1,144.00,0,2
4,147,2,,,RA,RA,RA,RA,RA,,,,RA,RA,RA,RA,,,,,,,,,,RA,RA,RA,RA,RA,,,,RA,RA,RA,RA,,,,RA,RA,RA,RA,RA,,,,,,,,,RA,RA,RA,RA,RA,,405.888393,28,0.0,0.00,0,2
5,148,1,,RA,RA,RA,RA,RA,,,,RA,RA,RA,RA,,,,RA,RA,RA,RA,RA,,,,,,,,,RA,RA,RA,RA,RA,,,,RA,RA,RA,RA,,,,RA,RA,RA,RA,RA,,,,,,,,,404.919643,28,0.0,0.00,0,1
6,37,0,{291SDF-,(DH DL)ATL-(DH DL)[DFW20.9],DFW-RFD-[DFW15.4],DFW-SDF-[OAK25.2],OAK-[ONT16.6],ONT-DEN-SDF},,,,,,,,,,,,,,,,{356SDF-,(DH AA)CLT-(DH AA)[DFW20.5],DFW-RFD-[DFW15.8],DFW-RFD-[DFW15.4],DFW-SDF-[OAK25.2],OAK-[ONT13.5],ONT-(DH DL)ATL-(DH DL)SDF},,,,,,,,{396SDF-(DH DL)DTW-,[LAN24.1],LAN-SDF-[LAN15.8],LAN-SDF-[LAN15.8],LAN-SDF-[LAN15.8],LAN-SDF-[LAN15.8],LAN-SDF-[EWR17.4],EWR-SDF},,,,,,,,{309SDF-(DH WN)MDW-(DH WN)[MCI20.5],MCI-SDF-[MKE15.3],MKE-SDF-[MKE15.3],MKE-SDF-[MKE15.3],MKE-SDF-[MKE15.3],MKE-SDF},,710.294639,28,62.5,140.05,0,0
7,38,0,{307SDF-,(DH WN)[ATL24.0],ATL-SDF-[CLE14.7],CLE-SDF-[CLE14.7],CLE-SDF-[CLE14.7],CLE-SDF-[CLE14.7],CLE-SDF},,,,,,,,{400SDF-(DH DL)ATL-(DH DL)[MDT27.8],—,MDT-SDF-[MDT15.2],MDT-SDF-[MDT15.2],MDT-SDF-[MDT15.2],MDT-SDF-[MDT15.9],MDT-SDF-[MDT11.6] MDT-,(DH AA)CLT-(DH AA)SDF},,,,,,,{326SDF-(DH DL)[DTW26],—,DTW-SDF-[ATL16.9],ATL-SDF-[ATL16.9],ATL-SDF-[ATL16.9],ATL-SDF-[MIA14.7],MIA-SDF},,{338SDF-(DH AA)[DFW14.8],DFW-[PHL31.6],PHL-,SDF-[ORD16.6],ORD-SDF-[MCO14.6],MCO-SDF-[CLE12.6] CLE-,(DH WN)MDW-(DH WN)SDF},,,,,,,,,,,,,,,708.584535,31,75.0,143.28,0,0
8,106,0,{395SDF-(DH DL)[ATL25.5],—,ATL-SDF-[ATL16.9],ATL-SDF-[ATL16.9],ATL-SDF-[ATL16.9],ATL-SDF-[MCO14.6],MCO-SDF-[EWR17.4],EWR-SDF},,,,,,,,,,,,,,{310SDF-,(DH AA)CLT-(DH AA)[RDU20.0],RDU-SDF-[RDU14.5],RDU-SDF-[RDU14.5],RDU-SDF-[RDU14.5],RDU-SDF-[RDU9.8] RDU-(DH AA)CLT-(DH AA)SDF},,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,,685.711538,13,75.0,74.40,0,0
9,105,0,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,VTO,,,,{600SDF-,[HNL20.6],HNL-(DH DL)ATL-(DH DL)SDF},,,,,,{600SDF-,[HNL20.6],HNL-(DH DL)ATL-(DH DL)SDF},,,,,{601SDF-,[HNL20.6],HNL-(DH DL)ATL-(DH DL)SDF},,,,,,"{612SDF-(C,DH UPS)CGN-

In [20]:
import re
import pandas as pd
from copy import copy
from datetime import date, datetime

from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Border, Side, Alignment
from openpyxl.worksheet.table import Table, TableStyleInfo
from openpyxl.utils import get_column_letter


def export_master_lines_to_excel_table(
    df,
    output_path,
    *,
    sheet_name="Lines",
    table_name="LinesTable",
    calendar_cols=None,
    training_start=None,
    training_end=None,
    vacation_ranges=None,
    table_style="TableStyleLight9",
    calendar_col_width=32,
    calendar_row_height=28,
    header_row_height=36,
    non_calendar_max_width=32,
):
    """
    Exports a pandas DataFrame to Excel as a real Excel table.

    Calendar formatting:
        - Calendar headers are displayed like: Wed, May 27
        - Any non-empty calendar cell is highlighted green.
        - Training start date gets thick red LEFT border.
        - Training end date gets thick red RIGHT border.
        - Vacation start date gets thick royal blue LEFT border.
        - Vacation end date gets thick royal blue RIGHT border.
        - Calendar columns all get the same fixed width.
        - Rows all get the same fixed height.

    vacation_ranges:
        Can be:
            [("2026-08-02", "2026-08-08")]
        or:
            [
                ("2026-08-02", "2026-08-08"),
                ("2026-08-16", "2026-08-22"),
            ]
    """

    df = df.copy()

    def normalize_date(value):
        if value is None or value == "":
            return None

        if isinstance(value, datetime):
            return value.date()

        if isinstance(value, date):
            return value

        try:
            return pd.to_datetime(value).date()
        except Exception:
            return None

    def format_calendar_header(d):
        # Cross-platform. Avoids "%-d", which does not work reliably on Windows.
        return f"{d:%a}, {d:%b} {d.day}"

    def sanitize_table_name(name):
        name = re.sub(r"\W+", "_", str(name))
        if not name:
            name = "Table1"
        if name[0].isdigit():
            name = "_" + name
        return name

    def normalize_date_ranges(ranges):
        """
        Accepts vacation ranges in either format:

            vacation_ranges=[
                {"start": "2023-05-01", "end": "2023-05-21"},
                {"start": "2023-06-01", "end": "2023-06-08"},
            ]

        Also still accepts the older tuple format:

            vacation_ranges=[
                ("2023-05-01", "2023-05-21"),
                ("2023-06-01", "2023-06-08"),
            ]
        """

        if not ranges:
            return []

        normalized = []

        for item in ranges:

            if isinstance(item, dict):
                start = item.get("start")
                end = item.get("end")

            else:
                try:
                    start, end = item
                except Exception:
                    continue

            start_date = normalize_date(start)
            end_date = normalize_date(end)

            if start_date is None or end_date is None:
                continue

            if end_date < start_date:
                start_date, end_date = end_date, start_date

            normalized.append((start_date, end_date))

        return normalized

    def visible_text_length(value):
        if value is None:
            return 0

        text = str(value)

        if "\n" in text:
            return max(len(part) for part in text.splitlines())

        return len(text)

    def autosize_column(ws, col_idx, min_width=8, max_width=30):
        max_len = 0

        for row in range(1, ws.max_row + 1):
            value = ws.cell(row=row, column=col_idx).value
            max_len = max(max_len, visible_text_length(value))

        width = max(min_width, max_len + 2)
        width = min(width, max_width)

        ws.column_dimensions[get_column_letter(col_idx)].width = width

    def add_border_side(cell, *, left=None, right=None):
        old = cell.border

        cell.border = Border(
            left=left if left is not None else copy(old.left),
            right=right if right is not None else copy(old.right),
            top=copy(old.top),
            bottom=copy(old.bottom),
            diagonal=copy(old.diagonal),
            diagonal_direction=old.diagonal_direction,
            diagonalUp=old.diagonalUp,
            diagonalDown=old.diagonalDown,
            outline=old.outline,
            vertical=copy(old.vertical),
            horizontal=copy(old.horizontal),
        )

    training_start = normalize_date(training_start)
    training_end = normalize_date(training_end)
    vacation_ranges = normalize_date_ranges(vacation_ranges)

    # Determine which DataFrame columns are calendar columns.
    if calendar_cols is None:
        calendar_date_set = {
            normalize_date(col)
            for col in df.columns
            if normalize_date(col) is not None
        }
    else:
        calendar_date_set = {
            normalize_date(col)
            for col in calendar_cols
            if normalize_date(col) is not None
        }

    # Rename calendar columns to visible Excel headers like "Wed, May 27".
    # Also keep an internal map from real date -> displayed header.
    new_columns = []
    date_to_display_header = {}
    display_header_to_date = {}

    for col in df.columns:
        col_date = normalize_date(col)

        if col_date in calendar_date_set:
            display_header = format_calendar_header(col_date)

            new_columns.append(display_header)
            date_to_display_header[col_date] = display_header
            display_header_to_date[display_header] = col_date
        else:
            new_columns.append(str(col))

    df.columns = new_columns

    # Write DataFrame
    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        df.to_excel(writer, sheet_name=sheet_name, index=False)

    wb = load_workbook(output_path)
    ws = wb[sheet_name]

    max_row = ws.max_row
    max_col = ws.max_column

    ws.freeze_panes = "A2"

    # Create real Excel Table
    table_name = sanitize_table_name(table_name)
    table_ref = f"A1:{get_column_letter(max_col)}{max_row}"

    tab = Table(displayName=table_name, ref=table_ref)

    style = TableStyleInfo(
        name=table_style,          # TableStyleLight9
        showFirstColumn=False,
        showLastColumn=False,
        showRowStripes=True,
        showColumnStripes=False,
    )

    tab.tableStyleInfo = style
    ws.add_table(tab)

    # Helps some spreadsheet programs recognize the filterable range.
    ws.auto_filter.ref = table_ref

    green_fill = PatternFill(
        fill_type="solid",
        fgColor="FFC6EFCE",
    )

    thick_red = Side(
        style="thick",
        color="FFFF0000",
    )

    royal_blue = Side(
        style="thick",
        color="FF4169E1",  # Royal Blue
    )

    # Build real-date -> Excel column index map from the displayed headers.
    date_to_excel_col = {}

    for col_idx in range(1, max_col + 1):
        header_value = ws.cell(row=1, column=col_idx).value

        if header_value in display_header_to_date:
            real_date = display_header_to_date[header_value]
            date_to_excel_col[real_date] = col_idx

    calendar_excel_cols = set(date_to_excel_col.values())

    # Fixed header row height
    ws.row_dimensions[1].height = header_row_height

    # Header formatting
    for cell in ws[1]:
        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True,
        )

    # Fixed row height for all data rows.
    # Note: Excel row height applies to the entire row, not only calendar cells.
    for row in range(2, max_row + 1):
        ws.row_dimensions[row].height = calendar_row_height

    # Calendar cell formatting
    for excel_col in calendar_excel_cols:
        col_letter = get_column_letter(excel_col)
        ws.column_dimensions[col_letter].width = calendar_col_width

        for row in range(2, max_row + 1):
            cell = ws.cell(row=row, column=excel_col)

            if cell.value not in (None, ""):
                cell.fill = green_fill

            cell.alignment = Alignment(
                horizontal="center",
                vertical="center",
                wrap_text=True,
            )

    # Training start: thick red LEFT border
    if training_start in date_to_excel_col:
        excel_col = date_to_excel_col[training_start]

        for row in range(1, max_row + 1):
            add_border_side(
                ws.cell(row=row, column=excel_col),
                left=thick_red,
            )

    # Training end: thick red RIGHT border
    if training_end in date_to_excel_col:
        excel_col = date_to_excel_col[training_end]

        for row in range(1, max_row + 1):
            add_border_side(
                ws.cell(row=row, column=excel_col),
                right=thick_red,
            )

    # Vacation start/end borders
    for vacation_start, vacation_end in vacation_ranges:

        # Vacation start: thick royal blue LEFT border
        if vacation_start in date_to_excel_col:
            excel_col = date_to_excel_col[vacation_start]

            for row in range(1, max_row + 1):
                add_border_side(
                    ws.cell(row=row, column=excel_col),
                    left=royal_blue,
                )

        # Vacation end: thick royal blue RIGHT border
        if vacation_end in date_to_excel_col:
            excel_col = date_to_excel_col[vacation_end]

            for row in range(1, max_row + 1):
                add_border_side(
                    ws.cell(row=row, column=excel_col),
                    right=royal_blue,
                )

    # Auto-size only the non-calendar columns
    for col_idx in range(1, max_col + 1):
        if col_idx in calendar_excel_cols:
            continue

        autosize_column(
            ws,
            col_idx,
            min_width=8,
            max_width=non_calendar_max_width,
        )

    wb.save(output_path)

In [30]:
import re
import pandas as pd
from copy import copy
from datetime import date, datetime, timedelta

from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Border, Side, Alignment, Font
from openpyxl.worksheet.table import Table, TableStyleInfo
from openpyxl.utils import get_column_letter


def export_master_lines_to_excel_table(
    df,
    output_path,
    *,
    sheet_name="Master Lines",
    table_name="MasterLinesTable",
    calendar_cols=None,
    training_start=None,
    training_end=None,
    vacation_ranges=None,
    table_style="TableStyleLight9",
    calendar_col_width=32,
    calendar_row_height=28,
    header_row_height=36,
    non_calendar_max_width=32,
):
    """
    Exports a pandas DataFrame to Excel as a real Excel table.

    Calendar formatting:
        - Calendar headers are displayed like: Wed, May 27
        - Any non-empty calendar cell is highlighted green.
        - Training start date gets thick solid red LEFT border.
        - Training end date gets dashed red RIGHT border.
        - Vacation start date gets thick solid royal blue LEFT border.
        - Vacation end date gets dashed royal blue RIGHT border.
        - Vacation date headers are royal blue.
        - Training date headers are red.
        - Training header color wins if it overlaps vacation.
        - Calendar columns all get the same fixed width.
        - Data rows all get the same fixed height.

    vacation_ranges format:
        vacation_ranges=[
            {"start": "2023-05-01", "end": "2023-05-21"},
            {"start": "2023-06-01", "end": "2023-06-08"},
        ]
    """

    df = df.copy()

    def normalize_date(value):
        if value is None or value == "":
            return None

        if isinstance(value, datetime):
            return value.date()

        if isinstance(value, date):
            return value

        try:
            return pd.to_datetime(value).date()
        except Exception:
            return None

    def format_calendar_header(d):
        # Cross-platform. Avoids "%-d", which does not work reliably on Windows.
        return f"{d:%a}, {d:%b} {d.day}"

    def sanitize_table_name(name):
        name = re.sub(r"\W+", "_", str(name))
        if not name:
            name = "Table1"
        if name[0].isdigit():
            name = "_" + name
        return name

    def normalize_date_ranges(ranges):
        """
        Accepts:
            [{"start": "2023-05-01", "end": "2023-05-21"}]

        Also accepts:
            [("2023-05-01", "2023-05-21")]
        """

        if not ranges:
            return []

        normalized = []

        for item in ranges:
            if isinstance(item, dict):
                start = item.get("start")
                end = item.get("end")
            else:
                try:
                    start, end = item
                except Exception:
                    continue

            start_date = normalize_date(start)
            end_date = normalize_date(end)

            if start_date is None or end_date is None:
                continue

            if end_date < start_date:
                start_date, end_date = end_date, start_date

            normalized.append((start_date, end_date))

        return normalized

    def visible_text_length(value):
        if value is None:
            return 0

        text = str(value)

        if "\n" in text:
            return max(len(part) for part in text.splitlines())

        return len(text)

    def autosize_column(ws, col_idx, min_width=8, max_width=30):
        max_len = 0

        for row in range(1, ws.max_row + 1):
            value = ws.cell(row=row, column=col_idx).value
            max_len = max(max_len, visible_text_length(value))

        width = max(min_width, max_len + 2)
        width = min(width, max_width)

        ws.column_dimensions[get_column_letter(col_idx)].width = width

    def add_border_side(cell, *, left=None, right=None):
        old = cell.border

        cell.border = Border(
            left=left if left is not None else copy(old.left),
            right=right if right is not None else copy(old.right),
            top=copy(old.top),
            bottom=copy(old.bottom),
            diagonal=copy(old.diagonal),
            diagonal_direction=old.diagonal_direction,
            diagonalUp=old.diagonalUp,
            diagonalDown=old.diagonalDown,
            outline=old.outline,
            vertical=copy(old.vertical),
            horizontal=copy(old.horizontal),
        )

    def date_in_any_range(d, ranges):
        return any(start <= d <= end for start, end in ranges)

    training_start = normalize_date(training_start)
    training_end = normalize_date(training_end)
    vacation_ranges = normalize_date_ranges(vacation_ranges)

    # Determine calendar columns.
    if calendar_cols is None:
        calendar_date_set = {
            normalize_date(col)
            for col in df.columns
            if normalize_date(col) is not None
        }
    else:
        calendar_date_set = {
            normalize_date(col)
            for col in calendar_cols
            if normalize_date(col) is not None
        }

    # Rename calendar columns to visible headers like "Wed, May 27".
    new_columns = []
    display_header_to_date = {}

    for col in df.columns:
        col_date = normalize_date(col)

        if col_date in calendar_date_set:
            display_header = format_calendar_header(col_date)

            new_columns.append(display_header)
            display_header_to_date[display_header] = col_date
        else:
            new_columns.append(str(col))

    df.columns = new_columns

    # Write DataFrame
    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        df.to_excel(writer, sheet_name=sheet_name, index=False)

    wb = load_workbook(output_path)
    ws = wb[sheet_name]

    max_row = ws.max_row
    max_col = ws.max_column

    ws.freeze_panes = "A2"

    # Create Excel table
    table_name = sanitize_table_name(table_name)
    table_ref = f"A1:{get_column_letter(max_col)}{max_row}"

    tab = Table(displayName=table_name, ref=table_ref)

    style = TableStyleInfo(
        name=table_style,  # TableStyleLight9
        showFirstColumn=False,
        showLastColumn=False,
        showRowStripes=True,
        showColumnStripes=False,
    )

    tab.tableStyleInfo = style
    ws.add_table(tab)

    # Helps some spreadsheet programs recognize the filterable range.
    ws.auto_filter.ref = table_ref

    # Fills
    green_fill = PatternFill(
        fill_type="solid",
        fgColor="FFC6EFCE",
    )

    vacation_header_fill = PatternFill(
        fill_type="solid",
        fgColor="FF800080",  # Royal blue
    )

    training_header_fill = PatternFill(
        fill_type="solid",
        fgColor="FFFFA500",  # Red
    )

    white_bold_font = Font(
        color="FFFFFFFF",
        bold=True,
    )

    # Borders
    training_start_side = Side(
        style="thick",
        color="FFFFA500",
    )

    training_end_side = Side(
        style="mediumDashed",
        color="FFFFA500",
    )

    vacation_start_side = Side(
        style="thick",
        color="FF800080",
    )

    vacation_end_side = Side(
        style="mediumDashed",
        color="FF800080",
    )

    # Build real-date -> Excel column index map.
    date_to_excel_col = {}

    for col_idx in range(1, max_col + 1):
        header_value = ws.cell(row=1, column=col_idx).value

        if header_value in display_header_to_date:
            real_date = display_header_to_date[header_value]
            date_to_excel_col[real_date] = col_idx

    calendar_excel_cols = set(date_to_excel_col.values())

    if date_to_excel_col:
        first_bid_date = min(date_to_excel_col)
        last_bid_date = max(date_to_excel_col)
    else:
        first_bid_date = None
        last_bid_date = None

    # Header row height
    ws.row_dimensions[1].height = header_row_height

    # Basic header formatting
    for cell in ws[1]:
        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True,
        )

    # Fixed row height for data rows.
    # Excel row height applies to the whole row.
    for row in range(2, max_row + 1):
        ws.row_dimensions[row].height = calendar_row_height

    # Calendar cell formatting
    for excel_col in calendar_excel_cols:
        col_letter = get_column_letter(excel_col)
        ws.column_dimensions[col_letter].width = calendar_col_width

        for row in range(2, max_row + 1):
            cell = ws.cell(row=row, column=excel_col)

            if cell.value not in (None, ""):
                cell.fill = green_fill

            cell.alignment = Alignment(
                horizontal="center",
                vertical="center",
                wrap_text=True,
            )

    # Color vacation headers royal blue
    for d, excel_col in date_to_excel_col.items():
        if date_in_any_range(d, vacation_ranges):
            header_cell = ws.cell(row=1, column=excel_col)
            header_cell.fill = vacation_header_fill
            header_cell.font = white_bold_font

    # Color training headers red.
    # This is done after vacation so training wins if there is overlap.
    if training_start is not None and training_end is not None:
        start = min(training_start, training_end)
        end = max(training_start, training_end)

        for d, excel_col in date_to_excel_col.items():
            if start <= d <= end:
                header_cell = ws.cell(row=1, column=excel_col)
                header_cell.fill = training_header_fill
                header_cell.font = white_bold_font

    # Training start: thick solid red LEFT border
    if training_start in date_to_excel_col:
        excel_col = date_to_excel_col[training_start]

        for row in range(1, max_row + 1):
            add_border_side(
                ws.cell(row=row, column=excel_col),
                left=training_start_side,
            )

    # Training end: dashed red RIGHT border
    if training_end in date_to_excel_col:
        excel_col = date_to_excel_col[training_end]

        for row in range(1, max_row + 1):
            add_border_side(
                ws.cell(row=row, column=excel_col),
                right=training_end_side,
            )

    # Vacation start/end borders inside the bid period
    for vacation_start, vacation_end in vacation_ranges:

        # Vacation start inside bid period:
        # thick solid royal blue LEFT border
        if vacation_start in date_to_excel_col:
            excel_col = date_to_excel_col[vacation_start]

            for row in range(1, max_row + 1):
                add_border_side(
                    ws.cell(row=row, column=excel_col),
                    left=vacation_start_side,
                )

        # Vacation end inside bid period:
        # dashed royal blue RIGHT border
        if vacation_end in date_to_excel_col:
            excel_col = date_to_excel_col[vacation_end]

            for row in range(1, max_row + 1):
                add_border_side(
                    ws.cell(row=row, column=excel_col),
                    right=vacation_end_side,
                )

        # Special case:
        # Vacation ended the day before the bid period starts.
        # Put dashed royal blue LEFT border on first bid date.
        if first_bid_date is not None:
            if vacation_end == first_bid_date - timedelta(days=1):
                excel_col = date_to_excel_col[first_bid_date]

                for row in range(1, max_row + 1):
                    add_border_side(
                        ws.cell(row=row, column=excel_col),
                        left=vacation_end_side,
                    )

        # Special case:
        # Vacation starts the day after the bid period ends.
        # Put thick solid royal blue RIGHT border on last bid date.
        if last_bid_date is not None:
            if vacation_start == last_bid_date + timedelta(days=1):
                excel_col = date_to_excel_col[last_bid_date]

                for row in range(1, max_row + 1):
                    add_border_side(
                        ws.cell(row=row, column=excel_col),
                        right=vacation_start_side,
                    )

    # Auto-size only the non-calendar columns
    for col_idx in range(1, max_col + 1):
        if col_idx in calendar_excel_cols:
            continue

        autosize_column(
            ws,
            col_idx,
            min_width=8,
            max_width=non_calendar_max_width,
        )

    wb.save(output_path)

In [31]:
export_master_lines_to_excel_table(dfsorted,
    "master_lines.xlsx",
    vacation_ranges=new_vacation_ranges,
    training_start="2023-06-01",
    training_end="2023-06-05",
)